# Load required packages

In [1]:
import pandas as pd
import numpy as np

# validation strategy 
from sklearn.model_selection import train_test_split 
from sklearn.model_selection import cross_val_score
from sklearn.utils import shuffle

# models
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression

# error measure/metrics
from sklearn.metrics import confusion_matrix 
from sklearn.metrics import classification_report

# Load Datasets - cleaned and one-hot encoded

In [2]:
# load epilepsy dataset 
epilepsy = pd.read_csv("./Datasets/OneHotEpilepsyCleaned.csv", sep= ',', decimal= '.', on_bad_lines = 'error', skip_blank_lines=False)

# load dementia dataset 
dementia = pd.read_csv("./Datasets/OneHotDementiaCleaned.csv", sep= ',', decimal= '.', on_bad_lines = 'error', skip_blank_lines=False)

# Check data loaded is correct

In [3]:
pd.set_option('display.max_columns', None)

In [4]:
epilepsy.head()

,Age,Weight,Height,Seizure Frequency,Seizure Duration,Epilepsy Type,Gender_Female,Gender_Male,Gender_Other,Medication Status_Not on Medication,Medication Status_On Medication,Alcohol or Drug Use_No,Alcohol or Drug Use_Yes,EEG Abnormality Detected_No,EEG Abnormality Detected_Yes,MRI/CT Scan Result_Abnormal,MRI/CT Scan Result_Normal,MRI/CT Scan Result_Not Done,Seizure Type_Absence,Seizure Type_Focal,Seizure Type_Generalized,Seizure Type_Tonic-Clonic,Aura Before Seizure_No,Aura Before Seizure_Yes,Loss of Consciousness_No,Loss of Consciousness_Yes,Muscle Stiffness_No,Muscle Stiffness_Yes,Jerky Movements_No,Jerky Movements_Yes,Postictal Confusion_No,Postictal Confusion_Yes,Blank Stare Episodes_No,Blank Stare Episodes_Yes,Eye Rolling_No,Eye Rolling_Yes,Stress or Anxiety Before Episode_No,Stress or Anxiety Before Episode_Yes,Lack of Sleep Before Episode_No,Lack of Sleep Before Episode_Yes,Flashing Lights Sensitivity_No,Flashing Lights Sensitivity_Yes,Loud Sound Sensitivity_No,Loud Sound Sensitivity_Yes,Missed Medication_No,Missed Medication_Yes,Family History of Epilepsy_No,Family History of Epilepsy_Yes,Head Injury History_No,Head Injury History_Yes,Brain Tumor_No,Brain Tumor_Yes,History of Stroke_No,History of Stroke_Yes,Genetic Disorder_No,Genetic Disorder_Yes,Developmental Delay (in Children)_No,Developmental Delay (in Children)_Yes
0,15,48.3,168.2,1,60,Generalised,False,True,False,False,True,False,True,False,True,False,True,False,False,False,False,True,True,False,False,True,True,False,False,True,False,True,True,False,False,True,False,True,False,True,False,True,False,True,True,False,True,False,True,False,True,False,True,False,True,False,False,True
1,36,44.8,156.2,3,30,Generalised,True,False,False,False,True,True,False,False,True,True,False,False,False,False,False,True,False,True,True,False,False,True,False,True,True,False,False,True,False,True,False,True,True,False,True,False,False,True,False,True,False,True,True,False,True,False,True,False,True,False,True,False
2,32,70.4,180.4,0,60,Focal,True,False,False,True,False,True,False,False,True,False,False,True,False,True,False,False,False,True,False,True,True,False,False,True,False,True,False,True,True,False,False,True,False,True,False,True,False,True,True,False,True,False,True,False,True,False,True,False,True,False,True,False
3,29,54.5,161.7,1,60,Generalised,False,True,False,False,True,True,False,False,True,False,False,True,False,False,False,True,True,False,False,True,False,True,False,True,False,True,True,False,True,False,False,True,True,False,True,False,False,True,True,False,False,True,True,False,False,True,True,False,True,False,True,False
4,18,62.1,168.0,1,30,Generalised,False,True,False,False,True,True,False,False,True,True,False,False,False,False,False,True,False,True,False,True,True,False,False,True,False,True,False,True,True,False,False,True,False,True,False,True,True,False,True,False,False,True,True,False,True,False,True,False,True,False,True,False


In [5]:
pd.reset_option('display.max_columns')

In [6]:
dementia.head()

,Group,Visit,Age,EDUC,SES,MMSE,CDR,eTIV,nWBV,ASF,M/F_F,M/F_M
0,Nondemented,1,87,14,2.0,27.0,0.0,1987,0.696,0.883,False,True
1,Nondemented,2,88,14,2.0,30.0,0.0,2004,0.681,0.876,False,True
2,Nondemented,1,88,18,3.0,28.0,0.0,1215,0.710,1.444,True,False
3,Nondemented,2,90,18,3.0,27.0,0.0,1200,0.718,1.462,True,False
4,Nondemented,1,80,12,4.0,28.0,0.0,1689,0.712,1.039,False,True


# Data Science Problems 

In this section, we will solve two classification problems. The problems are:
1. Epilepsy Type Classification with the Epilepsy Dataset
2. Dementia Group Classification with the Dementia Dataset

The Epilepsy Classification is a binary classification which consist of labels Focal and Generalised. The Dementia Classification is a multiclass classification which has three labels, non-demented, demented and converted. 

The machine learning models that will be used are Decision Tree and Logistic Regression The training steps are documented below. This includes validation strategy and evaluation methods. We will start with Decision Tree and move on with Logistic Regression

# Prepare data 

## Epilepsy Dataset

In [7]:
# get features data
X_epi = epilepsy.drop('Epilepsy Type', axis=1)
# get labels
y_epi = epilepsy['Epilepsy Type']

In [8]:
X_epi.head()

,Age,Weight,Height,Seizure Frequency,Seizure Duration,Gender_Female,Gender_Male,Gender_Other,Medication Status_Not on Medication,Medication Status_On Medication,...,Head Injury History_No,Head Injury History_Yes,Brain Tumor_No,Brain Tumor_Yes,History of Stroke_No,History of Stroke_Yes,Genetic Disorder_No,Genetic Disorder_Yes,Developmental Delay (in Children)_No,Developmental Delay (in Children)_Yes
0,15,48.3,168.2,1,60,False,True,False,False,True,...,True,False,True,False,True,False,True,False,False,True
1,36,44.8,156.2,3,30,True,False,False,False,True,...,True,False,True,False,True,False,True,False,True,False
2,32,70.4,180.4,0,60,True,False,False,True,False,...,True,False,True,False,True,False,True,False,True,False
3,29,54.5,161.7,1,60,False,True,False,False,True,...,True,False,False,True,True,False,True,False,True,False
4,18,62.1,168.0,1,30,False,True,False,False,True,...,True,False,True,False,True,False,True,False,True,False


In [11]:
# check shape of data 
print(f'data: {X_epi.shape}, label: {y_epi.shape}')

data: (621, 57), label: (621,)


In [13]:
X_train_epilepsy, X_test_epilepsy, y_train_epilepsy, y_test_epilepsy = train_test_split(X_epi, y_epi, test_size = 0.2, random_state = 0, stratify = y_epi)

In [14]:
# check shape of train and test data set 
print(f'X_train\t: {X_train_epilepsy.shape}, y_train\t: {y_train_epilepsy.shape}\nX_test\t: {X_test_epilepsy.shape}, y_test\t: {y_test_epilepsy.shape}')

X_train	: (496, 57), y_train	: (496,)
X_test	: (125, 57), y_test	: (125,)


## Dementia Dataset

In [17]:
# get features data
X_deme = dementia.iloc[:,1:]
# get labels
y_deme = dementia['Group']

In [18]:
# check shape of data 
print(f'data: {X_deme.shape}, label: {y_deme.shape}')

data: (354, 11), label: (354,)


In [19]:
X_train_dementia, X_test_dementia, y_train_dementia, y_test_dementia = train_test_split(X_deme, y_deme, test_size = 0.2, random_state = 0, stratify = y_deme)

In [20]:
# check shape of train and test data set 
print(f'X_train\t: {X_train_dementia.shape}, y_train\t: {y_train_dementia.shape}\nX_test\t: {X_test_dementia.shape}, y_test\t: {y_test_dementia.shape}')

X_train	: (283, 11), y_train	: (283,)
X_test	: (71, 11), y_test	: (71,)


# Decision Tree 

## Finding best parameters using cross validation
To prevent overfitting which is common for a decision tree, parameters 
- "max_depth"
- "min_samples_split"
- "min_samples_leaf"

will be fine tuned. To find the best set of parameters, the best average score across a Stratified 10-fold cross validation will be considered and unique pairs of {"max_depth", "min_sample_split", "min_samples_leaf"} will be used. 

Parameter criterion and max_features is set to default which uses 'gini impurity' and the square root of total no. of features respectively[1]. max_features is set to default because sqrt(16) = 4 which is 25% of the total no. of features in the dry bean dataset. The recommendation is to check up to 30-40%[2]. 

- [1] https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html
- [2] Practical data science week 4 slides

## Define a function to find best parameters for decision tree

In [21]:
def find_best_params_DT(X_train, y_train):
    # set of values for each parameter
    max_depth_vals = [5, 7, 10, 12, 15, 17]
    min_samples_split_vals = [2, 5, 10, 20, 50, 100]
    min_samples_leaf_vals = [1, 2, 3, 5, 10, 20]
    
    max_depth_his = list()
    min_samples_his = list()
    min_leafs_his = list()
    corr_avg = list() 
    
    for max_depth in max_depth_vals:
        for min_sample in min_samples_split_vals: 
            for min_leaf in min_samples_leaf_vals: 
                acc_scores = list() 
    
                # instance of Decision Tree for each parameter set
                dt_clf = DecisionTreeClassifier(max_depth = max_depth, min_samples_split = min_sample, min_samples_leaf = min_leaf)
                
                # get average scores of each k fold 
                acc_scores = cross_val_score(dt_clf, X_train, y_train, cv = 10, scoring = 'accuracy')
                
                # find average score across all folds 
                avg_score = acc_scores.mean()
        
                print(f'With max_depth={max_depth}, min_samples_split={min_sample}, and min_samples_leaf={min_leaf}, the average accurary is {avg_score}')
                
                max_depth_his.append(max_depth)
                min_samples_his.append(min_sample)
                min_leafs_his.append(min_leaf)
                corr_avg.append(avg_score)
    
    return max_depth_his, min_samples_his, min_leafs_his, corr_avg
        

## Epilepsy Type Classification

In [22]:
max_depth_his, min_samples_his, min_leafs_his, corr_avg = find_best_params_DT(X_train_epilepsy, y_train_epilepsy)

With max_depth=5, min_samples_split=2, and min_samples_leaf=1, the average accurary is 0.8869387755102041
With max_depth=5, min_samples_split=2, and min_samples_leaf=2, the average accurary is 0.8850204081632652
With max_depth=5, min_samples_split=2, and min_samples_leaf=3, the average accurary is 0.8889387755102041
With max_depth=5, min_samples_split=2, and min_samples_leaf=5, the average accurary is 0.882938775510204
With max_depth=5, min_samples_split=2, and min_samples_leaf=10, the average accurary is 0.8931020408163265
With max_depth=5, min_samples_split=2, and min_samples_leaf=20, the average accurary is 0.9132244897959184
With max_depth=5, min_samples_split=5, and min_samples_leaf=1, the average accurary is 0.8869387755102041
With max_depth=5, min_samples_split=5, and min_samples_leaf=2, the average accurary is 0.8870204081632653
With max_depth=5, min_samples_split=5, and min_samples_leaf=3, the average accurary is 0.8869387755102041
With max_depth=5, min_samples_split=5, and mi

In [23]:
max_ind = np.argmax(corr_avg)
best_max_depth = max_depth_his[max_ind]
best_min_samples_split = min_samples_his[max_ind]
best_min_leafs = min_leafs_his[max_ind]

print(f'The best acc is {corr_avg[max_ind]} with parameters:\n- max_depth\t\t=\t{best_max_depth}\n- min_samples_split\t=\t{best_min_samples_split}\n- min_samples_leaf\t=\t{best_min_leafs}')

The best acc is 0.9152653061224492 with parameters:
- max_depth		=	7
- min_samples_split	=	5
- min_samples_leaf	=	20


### Feature Selection/Hill climbing

Given Epilepsy dataset has 29 features, we can find the best set of features to train the final model along with the best set of paramters found. This is done using the hill climbing algorithm. Hill climbing randomly selects a feature from the dataset and adds it to a list where the features in the list is used to train the model. If the newly added feature improves the accuracy of the model, it is kept in the list. If it decreases or doesn't improve the accuracy, it is removed from the list.

This method works by having 19 different lists with range of values from 0 to n which represents each feature in the dataset and each list is randomly shuffled using 1 to 19 as the random_state parameter of the shuffle function[1]. It will go through all 19 of them to build the list that gives the best accuracy. This list will then be used to train the final model. Note that shuffle function may provide different shuffled lists when a kernel restarts. This affects the starting point and only moves towards a local minima rather than a global minimum.

- [1] https://scikit-learn.org/stable/modules/generated/sklearn.utils.shuffle.html

In [24]:
feas_set = list() 
scores = list() 

for i in range(1,20): 
    # to keep best features and score from them
    keep_feas_ind = []
    best_score = 0.0 
    
    # 16 features in epilepsy dataset
    fea_num = 29
    # create a new random list from the range of 1 to 15 by using a new random_state number 
    random_col_ind = shuffle(range(0,fea_num), random_state = i)

    print(f'random_state = {i}, col_ids = {random_col_ind}')
    for j in range(0, fea_num): 
        # add index of currently selected feature to the list
        keep_feas_ind.append(random_col_ind[j])
    
        # get bean features based off index listed in the list
        X_h = X_epi.iloc[:,keep_feas_ind]
    
        # create new train test split based off features in that list 
        X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(X_h, y_epi, test_size = 0.2, random_state = 0, stratify = y_epi)
    
        # create instace of KNN with the best k and p value
        dt_clf = DecisionTreeClassifier(max_depth = best_max_depth, min_samples_split = best_min_samples_split, min_samples_leaf = best_min_leafs)
    
        # fit data into model 
        dt_fit = dt_clf.fit(X_train_h, y_train_h)
        
        # test model with test dataset and get the score - accurary 
        dt_score = dt_fit.score(X_test_h, y_test_h)
    
        # if newly added feature doesn't improve the model based off score, remove it from list 
        if (dt_score < best_score) or (dt_score == best_score): 
            keep_feas_ind.remove(random_col_ind[j])
        else: 
            # else keep it and update the new best score
            best_score = dt_score
            print("Score with " + str(len(keep_feas_ind)) + " Selected features: " + str(dt_score))

    feas_set.append(keep_feas_ind)
    scores.append(best_score)
    print(keep_feas_ind)
    print()

random_state = 1, col_ids = [14, 21, 18, 20, 25, 19, 3, 10, 23, 22, 4, 2, 24, 6, 17, 13, 7, 26, 1, 16, 0, 15, 28, 27, 9, 8, 12, 11, 5]
Score with 1 Selected features: 0.584
Score with 2 Selected features: 0.888
[14, 18]

random_state = 2, col_ids = [1, 0, 14, 9, 20, 24, 16, 6, 3, 19, 27, 26, 12, 4, 10, 5, 21, 17, 2, 7, 25, 23, 18, 11, 22, 28, 13, 15, 8]
Score with 1 Selected features: 0.56
Score with 2 Selected features: 0.616
Score with 3 Selected features: 0.672
Score with 4 Selected features: 0.696
Score with 5 Selected features: 0.8
Score with 6 Selected features: 0.816
Score with 7 Selected features: 0.896
[1, 0, 24, 19, 27, 21, 18]

random_state = 3, col_ids = [18, 17, 12, 26, 15, 16, 13, 2, 1, 23, 14, 4, 22, 6, 7, 5, 20, 9, 11, 28, 19, 21, 0, 8, 27, 3, 25, 24, 10]
Score with 1 Selected features: 0.888
[18]

random_state = 4, col_ids = [11, 21, 27, 15, 20, 24, 17, 25, 19, 0, 3, 16, 10, 6, 12, 2, 4, 22, 13, 7, 9, 18, 28, 8, 1, 5, 23, 14, 26]
Score with 1 Selected features: 0.584
S

In [25]:
# get feature set with the highest score 
highest_score_ind = np.argmax(scores)
highest_score = scores[highest_score_ind]
best_features_index = feas_set[highest_score_ind]

print(f'The highest score is {highest_score}\nwith features: {X_epi.columns[best_features_index].tolist()}')

The highest score is 0.92
with features: ['Alcohol or Drug Use_Yes', 'Aura Before Seizure_No', 'Jerky Movements_No', 'Loss of Consciousness_Yes', 'Seizure Type_Focal', 'Weight']


In [26]:
# select the best features from the cleaned data
X_best_feas = X_epi.iloc[:, best_features_index]

# create new train test split based off best features
X_train_bf, X_test_bf, y_train_bf, y_test_bf = train_test_split(X_best_feas, y_epi, test_size = 0.2, random_state = 0, stratify = y_epi)

# create instace of KNN with the best max_depth, min_samples_split, and min_samples_leaf values
dt_clf = DecisionTreeClassifier(max_depth = best_max_depth, min_samples_split = best_min_samples_split, min_samples_leaf = best_min_leafs)

# fit data into model 
dt_fit = dt_clf.fit(X_train_bf, y_train_bf)

# predict with test data set 
dt_predicted_epilepsy = dt_fit.predict(X_test_bf)

## Dementia Type Classification

### Find the best parameters

In [27]:
max_depth_his, min_samples_his, min_leafs_his, corr_avg = find_best_params_DT(X_train_dementia, y_train_dementia)

With max_depth=5, min_samples_split=2, and min_samples_leaf=1, the average accurary is 0.8482758620689654
With max_depth=5, min_samples_split=2, and min_samples_leaf=2, the average accurary is 0.8305418719211822
With max_depth=5, min_samples_split=2, and min_samples_leaf=3, the average accurary is 0.8344827586206897
With max_depth=5, min_samples_split=2, and min_samples_leaf=5, the average accurary is 0.8483990147783251
With max_depth=5, min_samples_split=2, and min_samples_leaf=10, the average accurary is 0.862192118226601
With max_depth=5, min_samples_split=2, and min_samples_leaf=20, the average accurary is 0.8903940886699507
With max_depth=5, min_samples_split=5, and min_samples_leaf=1, the average accurary is 0.8584975369458128
With max_depth=5, min_samples_split=5, and min_samples_leaf=2, the average accurary is 0.8235221674876847
With max_depth=5, min_samples_split=5, and min_samples_leaf=3, the average accurary is 0.830911330049261
With max_depth=5, min_samples_split=5, and min

In [28]:
max_ind = np.argmax(corr_avg)
best_max_depth = max_depth_his[max_ind]
best_min_samples_split = min_samples_his[max_ind]
best_min_leafs = min_leafs_his[max_ind]

print(f'The best acc is {corr_avg[max_ind]} with parameters:\n- max_depth\t\t=\t{best_max_depth}\n- min_samples_split\t=\t{best_min_samples_split}\n- min_samples_leaf\t=\t{best_min_leafs}')

The best acc is 0.8903940886699507 with parameters:
- max_depth		=	5
- min_samples_split	=	2
- min_samples_leaf	=	20


Given there are only 10 features in the Dementia dataset, hill climbing will not be used here.

### Train with best set of parameters

In [29]:
# create instace of KNN with the best max_depth, min_samples_split, and min_samples_leaf values
dt_clf = DecisionTreeClassifier(max_depth = best_max_depth, min_samples_split = best_min_samples_split, min_samples_leaf = best_min_leafs)

# fit data into model 
dt_fit = dt_clf.fit(X_train_dementia, y_train_dementia)

# predict with test data set 
dt_predicted_dementia = dt_fit.predict(X_test_dementia)

## Evaluation with classification report

### Epilepsy Classification

In [30]:
# confusion matrix
print(confusion_matrix(y_test_epilepsy, dt_predicted_epilepsy))

print(classification_report(y_test_epilepsy, dt_predicted_epilepsy))

[[49  3]
 [ 7 66]]
              precision    recall  f1-score   support

       Focal       0.88      0.94      0.91        52
 Generalised       0.96      0.90      0.93        73

    accuracy                           0.92       125
   macro avg       0.92      0.92      0.92       125
weighted avg       0.92      0.92      0.92       125



### Dementia Classification

In [31]:
# confusion matrix
print(confusion_matrix(y_test_dementia, dt_predicted_dementia))

print(classification_report(y_test_dementia, dt_predicted_dementia))

[[ 0  1  6]
 [ 0 26  0]
 [ 0  1 37]]
              precision    recall  f1-score   support

   Converted       0.00      0.00      0.00         7
    Demented       0.93      1.00      0.96        26
 Nondemented       0.86      0.97      0.91        38

    accuracy                           0.89        71
   macro avg       0.60      0.66      0.63        71
weighted avg       0.80      0.89      0.84        71



/opt/anaconda3/envs/py3_12_7/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/anaconda3/envs/py3_12_7/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/anaconda3/envs/py3_12_7/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", 

#### ERROR NOTE:

Zero division warning error because the model has not classified a Converted correctly.

# Logistic Regression 

In [37]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score


## Dementia Dataset

In [138]:
lr_d = LogisticRegression(solver='lbfgs', l1_ratio=0, max_iter=20000, random_state=0)

In [139]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_dementia)
X_test_scaled = scaler.transform(X_test_dementia)

In [140]:
lr_d.fit(X_train_scaled, y_train_dementia)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",0
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multicla

In [141]:
y_pred_d = lr_d.predict(X_test_scaled)
y_prob_d = lr_d.predict_proba(X_test_scaled)  

In [142]:
print(f"Accuracy: {accuracy_score(y_test_dementia, y_pred_d):.2f}")
print("\nClassification Report:\n", classification_report(y_test_dementia, y_pred_d, target_names=dementia.Group.unique()))

Accuracy: 0.90

Classification Report:
               precision    recall  f1-score   support

 Nondemented       1.00      0.14      0.25         7
    Demented       0.96      1.00      0.98        26
   Converted       0.86      0.97      0.91        38

    accuracy                           0.90        71
   macro avg       0.94      0.71      0.71        71
weighted avg       0.91      0.90      0.87        71



In [128]:
y_prob_d

array([[4.31947897e-02, 4.62602964e-03, 9.52179181e-01],
       [4.40276681e-03, 9.95590659e-01, 6.57405717e-06],
       [4.69014091e-02, 1.24761245e-03, 9.51850978e-01],
       [5.38305312e-03, 4.51584653e-04, 9.94165362e-01],
       [3.43331926e-05, 9.99965608e-01, 5.89060438e-08],
       [6.87294798e-02, 3.70005581e-03, 9.27570464e-01],
       [4.18193117e-02, 9.33610245e-01, 2.45704428e-02],
       [1.32507316e-02, 1.40616886e-05, 9.86735207e-01],
       [5.81776313e-05, 9.99941785e-01, 3.78480493e-08],
       [3.88791614e-03, 9.95487043e-01, 6.25041300e-04],
       [4.21471086e-02, 9.54904590e-01, 2.94830164e-03],
       [2.28067751e-02, 2.13948575e-04, 9.76979276e-01],
       [3.92509290e-01, 5.38409774e-04, 6.06952301e-01],
       [1.32084936e-05, 9.99986755e-01, 3.69365225e-08],
       [5.59518776e-03, 4.04210653e-04, 9.94000602e-01],
       [2.92151636e-05, 9.99970712e-01, 7.28533465e-08],
       [3.11385792e-01, 6.62913444e-01, 2.57007643e-02],
       [1.28897463e-01, 8.63163

## Epilepsy Dataset

In [100]:
lr_e = LogisticRegression(solver='lbfgs', max_iter=600)

In [101]:
lr_e.fit(X_train_epilepsy, y_train_epilepsy)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [102]:
y_pred_e = lr_e.predict(X_test_epilepsy)
y_prob_e = lr_e.predict_proba(X_test_epilepsy) 

In [103]:
y_prob_e.shape
y_prob_e

array([[0.99263109, 0.00736891],
       [0.02655803, 0.97344197],
       [0.89652444, 0.10347556],
       [0.94670584, 0.05329416],
       [0.99341083, 0.00658917],
       [0.9256624 , 0.0743376 ],
       [0.93116323, 0.06883677],
       [0.01199094, 0.98800906],
       [0.9837261 , 0.0162739 ],
       [0.16425428, 0.83574572],
       [0.13771758, 0.86228242],
       [0.07297287, 0.92702713],
       [0.40737916, 0.59262084],
       [0.13169177, 0.86830823],
       [0.91762455, 0.08237545],
       [0.02226363, 0.97773637],
       [0.97848744, 0.02151256],
       [0.07806084, 0.92193916],
       [0.04786052, 0.95213948],
       [0.80420484, 0.19579516],
       [0.10269093, 0.89730907],
       [0.9323632 , 0.0676368 ],
       [0.99017249, 0.00982751],
       [0.01988233, 0.98011767],
       [0.98498545, 0.01501455],
       [0.24496073, 0.75503927],
       [0.96698439, 0.03301561],
       [0.83771906, 0.16228094],
       [0.01128457, 0.98871543],
       [0.98350426, 0.01649574],
       [0.

In [95]:
X_test_epilepsy.shape

(125, 57)